# 第四章：模型评估与正则化

## 从偏差-方差诊断到贝叶斯更新——理解模型为什么好、为什么坏

上一章用纯 NumPy 构建了神经网络并跑通了 MNIST。但模型训练完成后，真正的问题才刚开始：这个 85% 的准确率是好是坏？换一组数据会不会崩？怎么判断是数据不够还是模型太简单？

本章建立模型评估的完整框架：偏差-方差分解 → 交叉验证 → 正则化 → 分类指标 → 学习曲线诊断。在此基础上，引入贯穿新旧范式的底层原理——偏差-方差在现代 SFT/RAG/Agent 中的映射、正则化换了壳的多种形态、以及从 RLHF 到 Agent 的贝叶斯更新统一视角。

## 4.1 偏差-方差分解

期望预测误差的分解：

$$\mathbb{E}_{\mathcal{D}}\left[(y - \hat{f}_{\mathcal{D}}(x))^2\right]$$

- **偏差**：$\text{Bias}[\hat{f}] = \mathbb{E}[\hat{f}] - f$。模型族对真实函数的系统性偏离——由模型假设容量不足导致
- **方差**：$\text{Var}[\hat{f}] = \mathbb{E}[(\hat{f} - \mathbb{E}[\hat{f}])^2]$。模型对训练集采样的敏感性——由模型过度灵活导致
- **不可约误差** $\sigma^2$：数据生成过程固有的噪声

偏差和方差通过模型复杂度耦合：降低偏差（加参数）必然增加方差，反之亦然。最优复杂度在两者之间取得平衡。


## 4.1 偏差与方差分解

#### 偏差-方差分解的代数推导


### 4.1.1 底层原理：偏差-方差——穿越技术栈的通用诊断语言

偏差-方差分解不绑定任何算法。它提供的是一套**跨模型、跨范式的诊断框架**：任何模型的预测误差都可以拆成三块：

$$\mathbb{E}[(y - \hat{f}(x))^2] = \underbrace{(\mathbb{E}[\hat{f}(x)] - f(x))^2}_{\text{偏差}^2} + \underbrace{\mathbb{E}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]}_{\text{方差}} + \underbrace{\sigma^2}_{\text{不可约误差}}$$

- **偏差**：模型平均预测与真实值的差距。高偏差 = 模型太简单，学不到真实模式（欠拟合）。
- **方差**：模型对训练数据的敏感度。高方差 = 模型太复杂，记住了噪声而非模式（过拟合）。

#### 旧范式中的经典体现

| 场景 | 高偏差症状 | 高方差症状 |
|------|-----------|-----------|
| 多项式回归的次数选择 | 次数太低：拟合一条直线穿过抛物线数据 | 次数太高：曲线完美穿过每个样本点但剧烈震荡 |
| 决策树深度 | 深度太浅：分裂不足，无法捕捉复杂决策边界 | 深度太深：每个叶子只含一个样本，完全记忆训练集 |
| KNN (K-Nearest Neighbors) 的 K 值 | K 太大：决策边界过于平滑，细节信息丢失 | K=1：决策边界紧贴每个样本，噪声被当成信号 |

#### 同一个框架在新范式中的映射

**SFT (Supervised Fine-Tuning) 微调中的数据量决策**

用 LoRA (Low-Rank Adaptation) 对预训练模型做指令微调时：
- **数据太少**（如 100 条）→ 方差主导：模型对这 100 条拟合很好（训练 loss 低），但换个问法就崩
- **数据质量差**（噪声标注、格式不一致）→ 偏差主导：模型学到的"指令遵循模式"本身就有偏
- 增加高质量数据既降偏差又降方差——但数据收集有成本。偏差-方差框架帮你判断**当前瓶颈是数据量还是数据质量**：如果训练和验证 loss 差距大 → 方差问题，加数据量；如果两者都高且接近 → 偏差问题，改进数据质量或模型容量。

**RAG (Retrieval-Augmented Generation) 检索参数调优**

RAG 的检索 top-k 选择本质上是一个偏差-方差权衡：
- **top-k 太小**（如 k=1）→ 高方差：换一个检索问题可能返回完全不同的文档，回答不稳定
- **top-k 太大**（如 k=20）→ 高偏差：前几篇文档的信号被大量噪声文档淹没，回答泛化无力
- 最优 k 取决于文档库的噪声水平——干净知识库可以用较小的 k（信噪比高），开放域搜索需要较大的 k（覆盖更多可能性）

**Agent 决策中的探索-利用**

Agent 的 ReAct 循环中，每步需要决定"调用哪个工具"：
- 🀠 用当前最优工具 → 低偏差：每次都选已知最好的，但可能错过更好的路径
- 🀢 尝试不同工具 → 低方差：探索更多可能性，但可能浪费时间在无效路径上

这与强化学习中的探索-利用权衡在数学上等价——本质都是偏差-方差在新场景中的投影。

#### 为什么是底层原理

不管你做的是多项式回归、CNN (Convolutional Neural Network) 图像分类、还是 LLM Agent 编排，当你面对"这个模型/系统为什么不好"的问题时，你的诊断流程永远是：

1. 先看偏差：预测结果**系统性偏离**正确答案吗？（→ 调整模型容量、特征工程、数据质量）
2. 再看方差：同样的输入，不同训练/检索结果**波动大**吗？（→ 增加数据量、加正则、调 top-k）
3. 最后看不可约误差：剩下的误差能从数据本身解释吗？（→ 如果已经是理论下限，停止调参）

这个三步流程不随技术栈变化而变化。技术换了，诊断语言不变。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, KFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, \
    confusion_matrix, roc_curve, auc, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification, make_moons

# === 偏差-方差可视化 ===
np.random.seed(42)  # 固定随机种子，保证结果可复现

# 真实函数
def true_f(x):
    return np.sin(1.5 * np.pi * x)  # 返回结果

# 生成多个训练集，每个拟合一个模型
n_datasets = 20
n_samples = 30
x_test = np.linspace(0, 1, 200)  # 生成等间隔序列

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 创建子图网格
degrees = [1, 4, 15]
titles = ['High Bias (Underfit)', 'Good Fit', 'High Variance (Overfit)']

for ax, deg, title in zip(axes, degrees, titles):
    predictions = []
    for _ in range(n_datasets):
        X_train = np.random.uniform(0, 1, n_samples)  # 均匀分布随机数
        y_train = true_f(X_train) + np.random.normal(0, 0.2, n_samples)
        
        poly = PolynomialFeatures(degree=deg)
        X_poly = poly.fit_transform(X_train.reshape(-1, 1))  # 改变张量形状
        model = LinearRegression().fit(X_poly, y_train)  # 训练模型（拟合数据）
        
        X_test_poly = poly.transform(x_test.reshape(-1, 1))  # 改变张量形状
        pred = model.predict(X_test_poly)  # 用训练好的模型做预测
        predictions.append(pred)
        ax.plot(x_test, pred, alpha=0.15, color='blue', lw=0.5)
    
    mean_pred = np.mean(predictions, axis=0)  # 计算均值
    ax.plot(x_test, true_f(x_test), 'k-', lw=2, label='True f(x)')
    ax.plot(x_test, mean_pred, 'r-', lw=2, label='Mean model')
    
    # 方差区域
    std_pred = np.std(predictions, axis=0)  # 计算标准差
    ax.fill_between(x_test, mean_pred - std_pred, mean_pred + std_pred, alpha=0.2, color='red')
    
    ax.set_title(f'{title}\nDegree={deg}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/bias_variance.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("Key insight:")
print("  Degree 1: High bias — can't capture the sine curve shape")
print("  Degree 4: Good balance — captures the pattern without overfitting to noise")
print("  Degree 15: High variance — model follows every noise point, wildly different from true function")


## 4.2 交叉验证：为什么不能只分一次训练集和测试集

### 单次划分的问题

把数据随机切为 80% 训练 + 20% 测试，训练后得到 85% 准确率。换一个随机种子重新切，准确率变成了 82%。哪个数字是真实的？

单次划分把数据的随机波动和模型的真实性能混在一起。样本量越小，随机波动越大。同一个模型在同一个数据集上的不同随机划分，评估结果可能差 5 个百分点——这个波动完全来自划分的偶然性，不是模型本身的差异。

### K-Fold 交叉验证的原理

将数据均分为 $K$ 份。每次用 $K-1$ 份训练，留 1 份验证。重复 $K$ 次，每次留不同的那份做验证。最终报告 $K$ 次得分的均值和标准差：

$$\text{CV Score} = \frac{1}{K} \sum_{i=1}^{K} \text{Score}(\text{训练集} = \mathcal{D} \setminus \mathcal{D}_i, \text{验证集} = \mathcal{D}_i)$$

均值告诉你模型的预期性能，标准差告诉你性能有多稳定。标准差大意味着模型对数据划分敏感——可能某些子集包含关键样本而其他子集没有，提示你需要更多数据或更强的正则化。

### K 的选择

$K=5$ 或 $K=10$ 是常用值。$K$ 越大（留一法），每次训练的数据越多，评估越稳定，但计算量也越大（$K$ 次完整训练）。$K=5$ 在稳定性和成本之间取得平衡。

交叉验证的核心用途不是"证明模型好"，而是**选超参数**——在多个候选超参数中，选 CV 得分最高的那个。这是第四章最重要的实践：你有一个待选 alpha 值列表，对每个 alpha 跑 K-Fold CV，选 CV 均值最高的 alpha，然后用全部数据、该 alpha 训练最终模型。


In [ ]:
# === 交叉验证演示 ===
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge

X_diabetes, y_diabetes = load_diabetes(return_X_y=True)

# 不同 alpha 的 Ridge 回归 CV 比较
alphas = np.logspace(-3, 3, 20)  # 生成对数间隔序列
cv_scores_mean = []
cv_scores_std = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, X_diabetes, y_diabetes, cv=5,
                             scoring='neg_mean_squared_error')
    cv_scores_mean.append(-scores.mean())  # 沿指定维度求均值
    cv_scores_std.append(scores.std())

# 找最优 alpha
best_idx = np.argmin(cv_scores_mean)
best_alpha = alphas[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))  # 创建子图网格
ax.semilogx(alphas, cv_scores_mean, 'b-', lw=2, label='5-Fold CV MSE')
ax.fill_between(alphas,
                np.array(cv_scores_mean) - np.array(cv_scores_std),  # 创建 NumPy 数组
                np.array(cv_scores_mean) + np.array(cv_scores_std),  # 创建 NumPy 数组
                alpha=0.2, color='blue')
ax.axvline(best_alpha, color='red', linestyle='--',
           label=f'Best alpha={best_alpha:.2f}')
ax.axhline(cv_scores_mean[best_idx], color='green', linestyle='--',
           alpha=0.5, label=f'Best MSE={cv_scores_mean[best_idx]:.1f}')
ax.set_xlabel('Alpha (Regularization Strength)'); ax.set_ylabel('Mean Squared Error')
ax.set_title('Ridge Regression: 5-Fold Cross-Validation for Alpha Selection')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/cross_validation.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print(f"Best alpha: {best_alpha:.3f}, CV MSE: {cv_scores_mean[best_idx]:.1f}")
print(f"\nAlpha=0 (no regularization) MSE: {cv_scores_mean[0]:.1f} — worse!")
print("Cross-validation shows that some regularization IMPROVES generalization.")


## 4.3 正则化 (Regularization)

正则化通过在损失函数中增加**惩罚项**来限制模型复杂度，是防止过拟合的核心技术。

### L1 正则化 (Lasso)

$$

\mathcal{L}_{\text{L1}} = \mathcal{L}_{\text{original}} + \lambda \sum_{i=1}^{d} |w_i|

$$

- 产生**稀疏解**（某些权重精确为 0）→ 自动特征选择
- 适用于特征很多但只有少数重要的场景

### L2 正则化 (Ridge / Weight Decay)

$$

\mathcal{L}_{\text{L2}} = \mathcal{L}_{\text{original}} + \lambda \sum_{i=1}^{d} w_i^2

$$

- 让所有权重趋向于 0 但不为 0
- 对异常值更稳定

### Elastic Net

$$

\mathcal{L}_{\text{EN}} = \mathcal{L}_{\text{original}} + \lambda_1 \sum |w_i| + \lambda_2 \sum w_i^2

$$

### Dropout (深度学习专属)

训练时**随机将一部分神经元输出置 0**（概率 $p$），相当于每次训练一个不同的子网络：

测试时不 dropout，但将权重乘以 $(1-p)$ 以保持期望一致。

**直觉**：Dropout 迫使每个神经元学会"独立有用"的特征，而非依赖其他神经元——相当于对网络进行**集成学习**。


##### L1 为什么产生稀疏解：几何视角

L1 正则化将参数约束在菱形区域内，L2 约束在圆形区域内。对于二维参数 $(w_1, w_2)$：

```
        w₂                    w₂
        ↑                     ↑
        │    ╱ 等高线          │   ╱ 等高线
        │  ╱                  │ ╱
        │╱  ◇ 最优解          │╱  ○ 最优解
        └────────→ w₁         └────────→ w₁
         L1 (菱形)             L2 (圆形)
```

损失函数的等高线（椭圆）从外向内代表损失递减。参数在约束区域内寻找使损失最小的点——即**等高线首次接触约束边界的位置**。

L1 的菱形边界有**尖角**（坐标轴上）。当等高线与坐标轴上的尖角相交时，一个参数精确为 0——这就是稀疏性。L2 的圆形边界光滑无尖角，等高线几乎不会恰好在坐标轴上相切，因此不产生稀疏解。

推广到高维：L1 在 $2^d$ 个"尖角"（每个坐标轴上）处产生稀疏解。对于百万参数的神经网络，这意味着大量权重被精确推到零——等价于自动特征选择。


### 4.3.1 底层原理：正则化——"限制自由度"的通用策略

正则化的本质不是一个公式（L1/L2），而是一个思想：**在优化目标中引入额外的约束，防止模型走极端**。

$$\min_\theta \; \mathcal{L}(\theta) + \lambda \cdot R(\theta)$$

其中 $\mathcal{L}$ 是任务损失，$R$ 是正则化项，$\lambda$ 控制约束强度。但这个公式只是正则化的一种**显式实现**——当正则化换了一个壳，你还能认出它吗？

#### 旧范式的显式正则：L1/L2/Dropout

| 正则化 | 机制 | 约束的是什么 |
|--------|------|------------|
| **L1 (Lasso)** | 加绝对值惩罚 $\lambda|\theta|$ | 诱导稀疏解——大量权重精确为零 |
| **L2 (Ridge/Weight Decay)** | 加平方惩罚 $\lambda|\theta|^2$ | 所有权重都缩小但不为零 |
| **Dropout** | 随机丢弃神经元 | 防止神经元之间形成共适应（co-adaptation） |
| **Early Stopping** | 在验证误差回升前停止 | 限制优化迭代次数，间接限制模型复杂度 |

#### 新范式中的"换壳"正则——骨子里的同一个思想

**LoRA：硬约束——冻结的才是正则**

LoRA 不修改损失函数，而是直接限制**哪些参数可以改变**：

$$W = W_0 + \Delta W = W_0 + BA$$

大部分参数（$W_0$）完全冻结，只有低秩适配器（$BA$）可训练。这是正则化最极端的形式——不是"惩罚"大参数，而是"根本不允许"参数改变。从正则化角度看，LoRA 的 rank $r$ 就是它的 $\lambda$ 参数：$r$ 越小，约束越强，越不容易过拟合（但可能欠拟合）；$r$ 越大，约束越弱。

**数据筛选：最强的正则化是高质量数据**

当训练数据本身噪声低、覆盖全时，模型不需要额外的正则化项。OpenAI 的 SFT 实践表明：
- 1K 条精心筛选的高质量指令数据，效果优于 10K 条噪声数据
- 数据质量 = 隐式正则：干净数据天然限制了模型可以学的"错误模式"

这不是因为 L1/L2 原理变了，而是因为**数据筛选和 L2 正则解决的是同一个问题：防止模型学到噪声而非信号**。区别在于 L2 是在损失函数里加惩罚，数据筛选是在源头阻断噪声。

**DPO (Direct Preference Optimization) 的 $\beta$ 参数：隐式正则**

DPO 损失函数中的 $\beta$ 参数：

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\left(\beta\log\frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \beta\log\frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\right)$$

$\beta$ 控制的是**允许模型偏离参考模型的程度**。这不是 L2 正则，但它和 L2 正则的 $\lambda$ 在功能上完全等价——都是"限制模型改变的幅度"。$\beta$ 越小，越接近参考模型（强正则）；$\beta$ 越大，微调越激进（弱正则）。

#### 识别正则化的三个信号

无论具体技术怎么变，只要你想"能不能让模型学得少一点、稳一点"，你就在做正则化。判断一个操作是不是正则化的三个信号：

1. **是否引入额外的约束？** LoRA 冻结参数 → 是。增加训练数据 → 不是（数据是信号源，不是约束）。
2. **主要目的是否是防止过拟合？** Early stopping → 是。降低学习率 → 不完全是（可能只是为了收敛更稳定）。
3. **是否有可调节的强度参数？** L2 有 $\lambda$，LoRA 有 rank $r$，DPO 有 $\beta$ → 是。Dropout 有 rate → 是。

当你理解了这三个信号，你会发现：**RAG 里限制检索 top-k、Agent 里限制最大步数、prompt 里加"如果不知道请明确说"的指令——这些本质上都是正则化**，只不过约束的不是模型参数，而是模型的行为空间。


In [ ]:
# === L1 vs L2 对比 ===
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)  # 固定随机种子，保证结果可复现
n_features = 50
n_samples = 100

# 生成数据：只有前 5 个特征有用
X = np.random.randn(n_samples, n_features)  # 标准正态分布随机数
true_w = np.zeros(n_features)  # 创建全零数组
true_w[:5] = [3.0, -2.0, 1.5, -1.0, 0.5]
y = X @ true_w + np.random.randn(n_samples) * 0.5  # 标准正态分布随机数

models = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (L2, alpha=1)': Ridge(alpha=1.0),
    'Lasso (L1, alpha=0.1)': Lasso(alpha=0.1, max_iter=5000),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))  # 创建子图网格
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X, y)  # 训练模型（拟合数据）
    w = model.coef_ if hasattr(model, 'coef_') else model.coef_
    ax.stem(range(n_features), w, basefmt='k-')
    ax.stem(range(5), true_w[:5], linefmt='r-', markerfmt='ro', basefmt='k-')
    ax.set_title(f'{name}')
    ax.set_xlabel('Feature index'); ax.set_ylabel('Weight')
    nnz = np.sum(np.abs(w) > 1e-6)  # 求和
    ax.text(0.95, 0.95, f'Non-zero: {nnz}/{n_features}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10)

plt.suptitle('L1 (Lasso) produces SPARSE solution; L2 (Ridge) shrinks all weights')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/l1_l2_comparison.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 4.4 分类评估指标：准确率为什么不够

### 混淆矩阵的结构

二分类的四种预测结果：

| | 预测正 | 预测负 |
|---|:---:|:---:|
| **实际正** | TP (真正例) | FN (假负例) |
| **实际负** | FP (假正例) | TN (真负例) |

准确率 = (TP + TN) / (TP + TN + FP + FN)——一个数字，看起来直观。但一个极端例子能说明问题：

测试集 1000 个样本，正例仅 10 个（罕见病），负例 990 个。模型预测"所有人都是健康"——TP=0, FN=10, FP=0, TN=990。准确率 = 990/1000 = 99%。一个漏掉所有患者的模型拿到了 99% 的准确率。

### 精确率与召回率

**精确率 (Precision)** = TP / (TP + FP)：模型预测为"正"的样本中，有多少真的是正。衡量模型的预测有多精准。高精确率意味着模型不乱报——说这个人有病，大概率真的有病。

**召回率 (Recall)** = TP / (TP + FN)：所有真正的正例中，模型找到了多少。衡量模型有没有漏掉。高召回率意味着模型把大多数正例都找到了。

精确率和召回率相互制衡。把阈值调低（更容易判正），召回率升高（找到更多正例）但精确率降低（更多负例被判为正）。把阈值调高则相反。选择哪个指标取决于场景：

- **疾病筛查**：宁可误报不能漏诊 → 高召回率。漏掉一个患者代价远大于让一个健康人多做一次检查
- **垃圾邮件过滤**：宁可漏过不能误删 → 高精确率。误删一封重要邮件代价远大于收件箱多一封垃圾邮件

F1 分数是两者的调和平均：$F_1 = 2 \cdot \frac{P \cdot R}{P + R}$。调和平均（而非算术平均）惩罚极端不平衡——精确率 1.0、召回率 0.0 的 F1 是 0，不是 0.5。

### ROC 曲线与 AUC

ROC 曲线以假正率 (FPR = FP / (FP+TN)) 为横轴，真正率 (TPR = Recall = TP / (TP+FN)) 为纵轴，绘制分类阈值从低到高变化时的表现。

- 阈值为 0（全判正）：TPR=1, FPR=1 → 曲线右上角
- 阈值为 1（全判负）：TPR=0, FPR=0 → 曲线左下角
- 完美分类器：存在一个阈值使得 TPR=1 且 FPR=0 → 曲线紧贴左上角

**AUC (Area Under the Curve)** = ROC 曲线下的面积。AUC 的直观含义：随机取一个正例和一个负例，模型给正例的打分高于负例的概率。AUC=1.0 表示模型对所有正例的打分都高于所有负例（完美排序），AUC=0.5 表示随机猜测。

AUC 不依赖分类阈值——它衡量的是模型整体的排序能力而非某个阈值下的分类结果。对于不平衡数据，AUC 比准确率更能反映模型真实性能。


##### 准确率的陷阱：一个数字案例

考虑二分类问题，测试集 1000 个样本，其中正例仅 10 个（患病），负例 990 个（健康）。模型预测所有人都是"健康"：

| | 预测正 | 预测负 |
|---|---|---|
| 实际正 (10) | 0 (TP) | 10 (FN) |
| 实际负 (990) | 0 (FP) | 990 (TN) |

$$\text{Accuracy} = \frac{0 + 990}{1000} = 99\%$$

99% 准确率的模型——漏掉了所有真正的患者。这就是为什么仅看准确率是危险的。

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{0}{10} = 0\%$$
$$\text{Precision} = \frac{TP}{TP + FP} = \frac{0}{0}\ \text{(未定义)}$$

处理不平衡数据时，必须同时报告**精确率、召回率和 F1**。若正例极少（如欺诈检测、罕见病诊断），ROC-AUC 也可能给出乐观评估——此时 PR 曲线是更好的选择。


In [ ]:
# === 分类指标完整演示 ===
import matplotlib.pyplot as plt
X_cls, y_cls = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, weights=[0.7, 0.3],  # 不平衡 70:30
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.3, random_state=42)  # 将数据划分为训练集和测试集

# 训练模型
model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_tr, y_tr)  # 训练模型（拟合数据）
y_pred = model.predict(X_te)  # 用训练好的模型做预测
y_prob = model.predict_proba(X_te)[:, 1]

# 评估（重点：准确率的陷阱）
acc = accuracy_score(y_te, y_pred)
print(f"Accuracy: {acc:.3f}")
print(f"但如果全部预测为多数类 (class 0): {(y_te==0).mean():.3f}  ← 几乎一样!")  # 沿指定维度求均值
print(f"→ 在 70:30 不平衡下，准确率是误导性指标\n")

print(f"Precision: {precision_score(y_te, y_pred):.3f}  (预测为 '正' 的有多少真的为正)")
print(f"Recall:    {recall_score(y_te, y_pred):.3f}  (真实 '正' 的有多少被找出)")
print(f"F1:        {f1_score(y_te, y_pred):.3f}  (综合度量)")
print(f"\n完整报告:")
print(classification_report(y_te, y_pred, target_names=['Negative', 'Positive']))

# 混淆矩阵 + ROC
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 创建子图网格

# 混淆矩阵
cm = confusion_matrix(y_te, y_pred)
axes[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=20,
                     color='white' if cm[i, j] > cm.max()/2 else 'black')  # 沿指定维度取最大值
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred Neg', 'Pred Pos'])
axes[0].set_yticklabels(['True Neg', 'True Pos'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC 曲线
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', lw=1, label='Random (AUC=0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.2, color='blue')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/classification_metrics.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 4.5 学习曲线：诊断欠拟合与过拟合

学习曲线用训练集大小做横轴，训练误差和验证误差做纵轴。随着训练数据增多，两条曲线的形态揭示了模型的核心问题。

### 典型曲线形态

**高偏差（欠拟合）：** 训练误差和验证误差都高，且两者接近。
- 含义：模型太简单，不能捕捉数据的真实模式。增加更多数据不解决问题——模型本身容量不够。
- 解决：增加模型复杂度（更多层/更多神经元）、增加特征、减少正则化

**高方差（过拟合）：** 训练误差很低，验证误差很高，两者之间有很大差距。
- 含义：模型太灵活，记住了训练集中的噪声而非真实模式。增加数据有效——更多样本让模型无法记住所有噪声
- 解决：增加训练数据、增加正则化（Dropout/L2）、简化模型

**理想状态：** 两条曲线最终收敛到相近且可接受的误差水平。
- 含义：模型容量和训练数据量匹配。继续增加数据或调整模型容量的边际收益递减

### 学习曲线的实践用法

1. 用当前数据画学习曲线——如果验证误差仍在下降趋势中，说明更多数据可能有用
2. 如果验证误差已趋于平坦且与训练误差差距大，加数据是首要解法
3. 如果验证误差已趋于平坦且与训练误差差距不大，加模型容量是首要解法

学习曲线比单次训练/测试划分提供的信息多得多——它告诉你**瓶颈是数据量还是模型容量**，而不只是"当前准确率 85%"。


In [ ]:
# === 学习曲线可视化 ===
import numpy as np
import matplotlib.pyplot as plt
def plot_learning_curve(model, X, y, title, ax):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),  # 生成等间隔序列
        scoring='neg_mean_squared_error'
    )
    train_mean = -train_scores.mean(axis=1)  # 沿指定维度求均值
    train_std = train_scores.std(axis=1)
    val_mean = -val_scores.mean(axis=1)  # 沿指定维度求均值
    val_std = val_scores.std(axis=1)
    
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training error')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    ax.plot(train_sizes, val_mean, 'o-', color='red', label='Validation error')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='red')
    ax.set_xlabel('Training Set Size'); ax.set_ylabel('MSE')
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

X_lc, y_lc = load_diabetes(return_X_y=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # 创建子图网格
plot_learning_curve(LinearRegression(), X_lc, y_lc, 'Linear Regression (High Bias)', axes[0])

plot_learning_curve(
    Pipeline([('poly', PolynomialFeatures(degree=10)), ('lr', LinearRegression())]),
    X_lc, y_lc, 'Polynomial deg=10 (Overfit)', axes[1]
)

plot_learning_curve(
    Pipeline([('poly', PolynomialFeatures(degree=2)), ('ridge', Ridge(alpha=1.0))]),
    X_lc, y_lc, 'Polynomial deg=3 + L2 Reg (Good)', axes[2]
)

plt.suptitle('Learning Curves: Diagnosing Bias vs Variance')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/learning_curves.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

print("\nDiagnosis:")
print("  Left: Both errors high and converge → UNDERFITTING. Need more complexity.")
print("  Middle: Train error low, val error high with gap → OVERFITTING. Need regularization.")
print("  Right: Both errors converge to reasonable level → GOOD FIT.")


## 4.6 Early Stopping：用时间做正则化

### 原理

训练过程中，训练误差持续下降——参数持续拟合训练集。验证误差先降后升——过了某个点后，模型开始拟合训练集中的噪声而非真实模式。

Early Stopping 的做法：每个 epoch 后评估验证集。记录验证误差最低时的模型参数。若验证误差连续 $N$ 个 epoch 不下降，停止训练，恢复到验证误差最低点的参数。

### 为什么 Early Stopping 是正则化

正则化的本质是限制模型的有效复杂度。L1/L2 通过惩罚大权重限制复杂度。Dropout 通过随机丢弃神经元限制复杂度。Early Stopping 通过**限制优化步数**限制复杂度——越早停止，参数离初始值越近，模型的"有效自由度"越低。

从参数空间的角度看：梯度下降从初始点出发走向最小值。早停 = 在到达过拟合区域之前停下。这等价于对参数变化幅度施加隐式约束。

### 实践注意事项

- `patience` 参数（$N$）通常设为 5-20。太小的 patience 可能在验证误差的随机波动处误停，太大则浪费计算
- 保存验证误差最低的模型，而非最后一个 epoch 的模型——验证误差可能在下降-上升-下降中波动
- Early Stopping 可以和其他正则化（Dropout、L2）同时使用，效果叠加


## 4.7 底层原理：贝叶斯视角——不确定性量化的统一语言

贝叶斯公式是处理不确定性的底层框架，但它远不只存在于朴素贝叶斯分类器中：

$$P(H|D) = \frac{P(D|H) \cdot P(H)}{P(D)}$$

核心思想：**先有先验信念（prior），看到证据后更新信念（posterior）**。这个更新结构穿越了 AI 的整个技术栈。

### 旧范式中的贝叶斯：显式概率模型

| 方法 | 先验 $P(H)$ | 似然 $P(D|H)$ | 后验 $P(H|D)$ |
|------|-----------|-------------|-------------|
| 朴素贝叶斯 | 类别的先验概率 | 给定类别下，特征取值的概率 | 给定特征后，类别的概率 |
| GMM (高斯混合模型) | 每个高斯分量的权重 | 给定分量下，样本的生成概率 | 给定样本后，分量的归属概率（EM 算法的 E 步） |
| LDA (Linear Discriminant Analysis) (线性判别分析) | 类别的先验概率 | 每类特征的多元高斯分布 | 后验分类概率 |

这些方法的共同局限：**需要显式建模概率分布**。当数据维度高、分布复杂时，显式建模要么不可行，要么过于粗糙。

### 新范式中的贝叶斯：隐式先验-证据更新

现代方法几乎从不显式写出 $P(H|D) \propto P(D|H) \cdot P(H)$，但它们的结构骨子里就是贝叶斯更新。

**RLHF (Reinforcement Learning from Human Feedback)：奖励模型训练 → PPO 策略更新**

RLHF 整体管线是一个贝叶斯更新过程：

| 阶段 | 贝叶斯角色 | 具体含义 |
|------|----------|---------|
| 预训练模型 | **先验** $P(H)$ | 模型对人类语言和偏好的初始理解——在没看到任何人类反馈之前的"信念" |
| 奖励模型训练 | **似然函数** $P(D|H)$ | 给定当前策略（模型行为），人类偏好数据出现的概率——"如果模型是这样想的，人类会怎么打分" |
| PPO 更新后的策略 | **后验** $P(H|D)$ | 看到人类偏好数据后，更新过的模型行为——"综合了先验知识和新证据后的最佳策略" |

**DPO (Direct Preference Optimization)：绕过显式后验的隐式贝叶斯**

DPO 没有显式先验/后验的数学形式，但它的核心机制是贝叶斯的：
- 参考模型 $\pi_{\text{ref}}$ = 先验信念（"微调之前的模型"）
- 偏好数据 = 证据（"人类更认可 chosen 而非 rejected"）
- 优化后的 $\pi_\theta$ = 后验信念（"综合先验与证据后的最佳模型"）

DPO 的 $\beta$ 参数控制的是**先验的权重**：$\beta$ 越小，先验（参考模型）的权重越大，更新越保守——这等价于贝叶斯更新中"先验很强时，需要更多证据才能改变信念"。

**RAG：检索 = 似然更新**

RAG 的运行逻辑就是一个实时贝叶斯更新：

| 组件 | 贝叶斯角色 |
|------|----------|
| LLM 的参数化知识（来自预训练） | **先验信念**：模型在没看到外部文档之前"认为"的答案 |
| 检索到的文档 | **证据/似然**：外部知识库告诉模型的"事实" |
| 基于文档生成的回答 | **后验**：综合了先验知识 + 外部证据后的最终输出 |

当检索到的文档与模型参数化知识冲突时（"我记忆中答案是 A，但检索到的文档说是 B"），LLM 需要做一个贝叶斯式决策：信自己的参数记忆（强先验）还是信检索文档（强证据）？高质量的 prompt 设计（如"基于参考文档回答，如无信息请说明"）本质上是在调**先验和证据的相对权重**。

**Agent：ReAct 循环 = 序列贝叶斯更新**

Agent 的每一步 Thought → Action → Observation 循环:

| 步骤 | 贝叶斯角色 |
|------|----------|
| 当前 Thought（执行 Action 之前的信念） | **当前先验** |
| Observation（工具返回的结果） | **新证据** |
| 下一轮 Thought（基于 Observation 调整后的决策） | **更新后的后验** |

Agent 的整个执行轨迹就是一个**序列贝叶斯更新**——每一步看到新证据（Observation）后更新对任务状态的信念（Thought），然后基于新信念选择下一步行动。

### 为什么必须用贝叶斯视角理解现代 AI

不用贝叶斯视角，你会看到一堆不相关的技术组件：
- RLHF 里的 PPO、DPO、$\beta$ 参数、参考模型...
- RAG 里的检索器、top-k、prompt 模板、幻觉问题...
- Agent 里的 ReAct 循环、工具调用、Observation 反馈...

用贝叶斯视角，你看到的是一个统一结构：**任何 AI 系统在接收新信息后更新内部状态的逻辑，本质上都是贝叶斯更新**。区别只在于：
- 旧范式：显式对概率建模（朴素贝叶斯、GMM）
- 新范式：隐式贝叶斯——先验/证据/后验不写公式，但系统结构就是按这个逻辑运行的


## 本章小结

### 技术层

| 概念 | 一句话总结 |
|------|-----------|
| **Bias-Variance** | 误差 = 偏差² + 方差 + 噪声；偏差和方差不可兼得 |
| **Cross-Validation** | 多次划分取平均，减少单次划分的随机性 |
| **L1/L2/Dropout** | L1 诱导稀疏，L2 均匀收缩，Dropout 防共适应 |
| **ROC (Receiver Operating Characteristic)-AUC** | 分类阈值无关的评估指标，不受类别不平衡影响 |
| **学习曲线** | 训练/验证误差的差距 = 过拟合程度；两者都高 = 欠拟合 |
| **Early Stopping** | 最简单有效的正则化——在验证误差最低点停下 |

### 底层原理层

| 原理 | 一句话 |
|------|--------|
| **偏差-方差诊断框架** | 不绑定算法——做 SFT 时判断瓶颈是数据量还是数据质量，调 RAG top-k 时平衡检索精度和覆盖，做 Agent 时权衡探索与利用，都在用同一个框架 |
| **正则化——限制自由度** | L1/L2/Dropout 是显式正则，LoRA 的冻结参数、高质量数据筛选、DPO 的 β 参数、RAG 的 top-k 限制是隐式正则——换了壳，同一个思想 |
| **贝叶斯视角** | RLHF 是先验→证据→后验的隐式贝叶斯更新，DPO 的参考模型是贝叶斯先验，RAG 的检索文档是贝叶斯证据，Agent 的 ReAct 循环是序列贝叶斯更新——技术栈不同，信念更新的逻辑相同 |

> 这五个底层原理（偏差-方差、正则化、贝叶斯更新、梯度优化、信号分类）构成 AI 工程师的元认知框架。具体算法会过时，但这五个框架穿越技术栈，换了名字仍然成立。
